# LSTM V4 — ưu tiên dự báo phiên kế tiếp

Notebook tự chứa runtime/script/protocol V4. **Chỉ attach dataset V3 hiện có** với `vci_ohlcv_lstm_v3.csv` và `dataset.manifest.json`, không cần ZIP/.bin hay script mới trong dataset. Không thay V2/V3.

Bật GPU rồi Run All. Mặc định pilot3epochs; đổi RUN_MODE='full' để train3folds ×2seeds, max35epochs/trial. Kết quả lịch sử đã xem chỉ là DEVELOPMENT; không ép pass hoặc tự tích hợp vào app. T4×2 không tự thành distributed training. Không pip install hay tải model khác.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import os, sys, json, hashlib, zipfile, runpy, shutil

RUN_MODE = 'pilot'  # 'full' = 3 folds x 2 seeds
VARIANT = 'priority_multitask'  # ablation tùy chọn: 'regression_only', output riêng
EXPLICIT_CSV = ''  # chỉ điền nếu attach nhiều CSV V3
RESUME_DIR = ''  # run cũ có run-config.json; giữ nguyên mode/variant/code/dataset

if os.name == 'nt':
    raise RuntimeError('Chạy notebook GPU trên Kaggle/Colab, không train từ app Windows.')
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/content/stock-agent-v3')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
if RUN_MODE not in {'pilot', 'full'} or VARIANT not in {'priority_multitask', 'regression_only'}:
    raise ValueError('Sai RUN_MODE hoặc VARIANT.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:8]
print('V4 mode:', RUN_MODE, '| variant:', VARIANT, '| run:', RUN_ID)


In [ ]:
PORTABLE_SOURCES = {"stock_multi_agent/services/ml_features_v3.py":"\"\"\"Causal, single-ticker V3 features shared by training and serving.\n\nRows are retained through warm-up. Eligibility applies to window endpoints, never\nto future returns. No cross-sectional proxy is silently substituted at serving.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nFEATURE_VERSION = \"single-ticker-v3a-1\"\nFEATURES = [\n    \"return_1\", \"return_2\", \"return_5\", \"return_10\", \"return_20\",\n    \"open_gap\", \"intraday_range\", \"close_position\", \"log_volume\",\n    \"volume_ratio_5\", \"volume_ratio_20\", \"ma_distance_5\", \"ma_distance_20\",\n    \"ma_distance_60\", \"volatility_5\", \"volatility_20\", \"active_ratio_20\",\n]\nHORIZONS = (1, 5)\n\n\ndef build_feature_frame(raw: pd.DataFrame, *, max_gap_business_days: int = 5) -> pd.DataFrame:\n    \"\"\"Use only information at/before each row; future rows cannot change features.\n\n    ``segment_id`` from the builder is preserved. Without it, unverified gaps\n    exceeding five weekdays create segments; weekends are not fabricated bars.\n    This conservative rule is not an exchange-holiday calendar certification.\n    \"\"\"\n    frame = raw.copy()\n    frame[\"date\"] = pd.to_datetime(frame[\"date\"]).dt.tz_localize(None).dt.normalize()\n    frame = frame.sort_values([\"ticker\", \"date\"]).reset_index(drop=True)\n    if frame.duplicated([\"ticker\", \"date\"]).any():\n        raise ValueError(\"Duplicate ticker/date rows must be reconciled before feature construction\")\n    if \"segment_id\" not in frame:\n        gap = frame.groupby(\"ticker\")[\"date\"].diff().dt.days.fillna(0)\n        previous = frame.groupby(\"ticker\")[\"date\"].shift(1)\n        weekdays = np.zeros(len(frame), dtype=np.int64)\n        present = previous.notna()\n        weekdays[present] = np.busday_count(\n            previous[present].to_numpy(dtype=\"datetime64[D]\"),\n            frame.loc[present, \"date\"].to_numpy(dtype=\"datetime64[D]\"),\n        )\n        breaks = frame[\"ticker\"].ne(frame[\"ticker\"].shift()) | (weekdays > max_gap_business_days) | (gap < 0)\n        frame[\"segment_id\"] = breaks.cumsum().astype(int)\n    group = frame.groupby([\"ticker\", \"segment_id\"], sort=False)\n    previous_close = group[\"close\"].shift(1)\n    for period in (1, 2, 5, 10, 20):\n        frame[f\"return_{period}\"] = group[\"close\"].pct_change(period, fill_method=None)\n    frame[\"open_gap\"] = frame[\"open\"] / previous_close - 1\n    frame[\"intraday_range\"] = frame[\"high\"] / frame[\"low\"] - 1\n    spread = frame[\"high\"] - frame[\"low\"]\n    frame[\"close_position\"] = ((frame[\"close\"] - frame[\"low\"]) / spread.replace(0, np.nan)).where(spread.ne(0), 0.5)\n    frame[\"log_volume\"] = np.log1p(frame[\"volume\"])\n    for period in (5, 20):\n        volume_mean = group[\"volume\"].transform(lambda values: values.rolling(period, min_periods=period).mean())\n        frame[f\"volume_ratio_{period}\"] = frame[\"volume\"] / volume_mean.replace(0, np.nan)\n    for period in (5, 20, 60):\n        price_mean = group[\"close\"].transform(lambda values: values.rolling(period, min_periods=period).mean())\n        frame[f\"ma_distance_{period}\"] = frame[\"close\"] / price_mean - 1\n    for period in (5, 20):\n        frame[f\"volatility_{period}\"] = frame.groupby([\"ticker\", \"segment_id\"])[\"return_1\"].transform(\n            lambda values: values.rolling(period, min_periods=period).std(ddof=1)\n        )\n    frame[\"active_ratio_20\"] = group[\"volume\"].transform(lambda values: values.gt(0).rolling(20, min_periods=20).mean())\n    finite = np.isfinite(frame[FEATURES].to_numpy(dtype=float)).all(axis=1)\n    frame[\"feature_valid\"] = finite\n    frame[\"endpoint_eligible\"] = finite & frame[\"volume\"].gt(0) & frame[\"active_ratio_20\"].ge(0.5)\n    return frame\n\n\ndef add_targets(frame: pd.DataFrame, horizons: tuple[int, ...] = HORIZONS) -> pd.DataFrame:\n    \"\"\"Create labels within segments, never use labels to select liquid endpoints.\"\"\"\n    result = frame.copy()\n    group = result.groupby([\"ticker\", \"segment_id\"], sort=False)\n    for horizon in horizons:\n        result[f\"target_return_{horizon}\"] = group[\"close\"].shift(-horizon) / result[\"close\"] - 1\n        result[f\"target_end_{horizon}\"] = group[\"date\"].shift(-horizon)\n    result[\"forecast_origin\"] = result[\"date\"]\n    result[\"feature_end\"] = result[\"date\"]\n    return result\n\n\ndef window_endpoints(frame: pd.DataFrame, window: int = 30, *, require_targets: bool = True) -> np.ndarray:\n    \"\"\"Return positional endpoints with contiguous, finite same-segment context.\"\"\"\n    if window < 1:\n        raise ValueError(\"window must be positive\")\n    valid = frame[\"feature_valid\"].astype(int)\n    context = valid.groupby([frame[\"ticker\"], frame[\"segment_id\"]]).transform(\n        lambda values: values.rolling(window, min_periods=window).sum().eq(window)\n    )\n    eligible = context & frame[\"endpoint_eligible\"]\n    if require_targets:\n        eligible &= np.isfinite(frame[[f\"target_return_{h}\" for h in HORIZONS]].to_numpy(dtype=float)).all(axis=1)\n        eligible &= frame[[f\"target_end_{h}\" for h in HORIZONS]].notna().all(axis=1)\n    return np.flatnonzero(eligible.to_numpy())\n\n\ndef window_batch(values: np.ndarray, endpoints: np.ndarray, window: int) -> np.ndarray:\n    \"\"\"Allocate only one batch, not N x window x features for the full dataset.\"\"\"\n    offsets = np.arange(window - 1, -1, -1)\n    return values[np.asarray(endpoints)[:, None] - offsets[None, :]]\n","stock_multi_agent/services/ml_validation.py":"\"\"\"Expanding, purged chronological validation and train-only transforms.\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nfrom stock_multi_agent.services.ml_features_v3 import FEATURES\n\n\n@dataclass\nclass WalkForwardFold:\n    fold_id: int\n    train: np.ndarray\n    validation: np.ndarray\n    calibration: np.ndarray\n    test: np.ndarray\n    boundaries: dict[str, str]\n\n    def manifest(self) -> dict[str, Any]:\n        return {\"fold_id\": self.fold_id, **self.boundaries, \"counts\": {name: int(len(getattr(self, name))) for name in (\"train\", \"validation\", \"calibration\", \"test\")}}\n\n\ndef expanding_purged_folds(frame: pd.DataFrame, endpoints: np.ndarray, *, folds: int = 3, gap_sessions: int = 5) -> list[WalkForwardFold]:\n    \"\"\"Each endpoint belongs to at most one outer test; labels cannot cross splits.\n\n    The last target horizon is used for purging. Gap counts use observed market\n    dates (sessions), not calendar days. Validation is split into model-selection\n    and a later, disjoint calibration block, each purged at its label boundary.\n    \"\"\"\n    if folds < 1 or gap_sessions < 0:\n        raise ValueError(\"folds must be positive and gap_sessions nonnegative\")\n    endpoints = np.asarray(endpoints, dtype=np.int64)\n    if not len(endpoints):\n        raise ValueError(\"No endpoints\")\n    origins = pd.to_datetime(frame.iloc[endpoints][\"date\"]).to_numpy(dtype=\"datetime64[ns]\")\n    target_end = pd.to_datetime(frame.iloc[endpoints][\"target_end_5\"]).to_numpy(dtype=\"datetime64[ns]\")\n    # Session gaps count the observed source calendar, not merely dates with\n    # liquid/finite endpoints. Otherwise sparse eligibility compresses time.\n    dates = np.sort(pd.to_datetime(frame[\"date\"]).dropna().unique().to_numpy(dtype=\"datetime64[ns]\"))\n    if len(dates) < max(80, 20 * folds + gap_sessions * 4):\n        raise ValueError(\"Too few distinct observed sessions for purged walk-forward folds\")\n    # Initial 45% training; remaining sessions divided into validation/test blocks.\n    initial = int(len(dates) * 0.45)\n    block = (len(dates) - initial) // (folds * 2)\n    result: list[WalkForwardFold] = []\n    for number in range(folds):\n        train_end_i = initial + number * 2 * block - 1\n        val_start_i = train_end_i + gap_sessions + 1\n        val_end_i = initial + (number * 2 + 1) * block - 1\n        cal_start_i = val_start_i + (val_end_i - val_start_i) // 2 + gap_sessions\n        validation_end_i = cal_start_i - gap_sessions - 1\n        test_start_i = val_end_i + gap_sessions + 1\n        test_end_i = len(dates) - 1 if number == folds - 1 else initial + (number * 2 + 2) * block - 1\n        if not (val_start_i <= validation_end_i < cal_start_i <= val_end_i < test_start_i <= test_end_i):\n            raise ValueError(\"Not enough sessions for separate selection/calibration/test blocks; collect longer history\")\n        masks = {\n            \"train\": (origins <= dates[train_end_i]) & (target_end <= dates[train_end_i]),\n            \"validation\": (origins >= dates[val_start_i]) & (target_end <= dates[validation_end_i]),\n            \"calibration\": (origins >= dates[cal_start_i]) & (target_end <= dates[val_end_i]),\n            \"test\": (origins >= dates[test_start_i]) & (origins <= dates[test_end_i]) & (target_end <= dates[test_end_i]),\n        }\n        indices = {name: endpoints[mask] for name, mask in masks.items()}\n        if any(len(value) == 0 for value in indices.values()):\n            raise ValueError(\"Empty purged split; add history or reduce fold count before registering protocol\")\n        boundaries = {\"train_end\": str(pd.Timestamp(dates[train_end_i]).date()), \"validation_start\": str(pd.Timestamp(dates[val_start_i]).date()), \"validation_end\": str(pd.Timestamp(dates[validation_end_i]).date()), \"calibration_start\": str(pd.Timestamp(dates[cal_start_i]).date()), \"calibration_end\": str(pd.Timestamp(dates[val_end_i]).date()), \"test_start\": str(pd.Timestamp(dates[test_start_i]).date()), \"test_end\": str(pd.Timestamp(dates[test_end_i]).date()), \"gap_sessions\": str(gap_sessions)}\n        result.append(WalkForwardFold(number, **indices, boundaries=boundaries))\n    return result\n\n\nclass TrainOnlyScaler:\n    def fit(self, values: np.ndarray, clip_quantile: float = 0.005) -> \"TrainOnlyScaler\":\n        if not 0 <= clip_quantile < 0.5:\n            raise ValueError(\"clip_quantile must be in [0, 0.5)\")\n        values = np.asarray(values, dtype=float)\n        if values.ndim != 2 or not len(values) or not np.isfinite(values).all():\n            raise ValueError(\"Scaler requires a finite nonempty train matrix\")\n        self.lower = np.quantile(values, clip_quantile, axis=0)\n        self.upper = np.quantile(values, 1 - clip_quantile, axis=0)\n        clipped = np.clip(values, self.lower, self.upper)\n        self.mean = clipped.mean(axis=0)\n        self.std = clipped.std(axis=0)\n        self.std[self.std < 1e-8] = 1\n        return self\n\n    def transform(self, values: np.ndarray) -> np.ndarray:\n        return ((np.clip(values, self.lower, self.upper) - self.mean) / self.std).astype(np.float32)\n\n    def specification(self) -> dict:\n        return {\"features\": FEATURES, \"mean\": self.mean.tolist(), \"std\": self.std.tolist(), \"lower\": self.lower.tolist(), \"upper\": self.upper.tolist(), \"fit_scope\": \"TRAIN_ONLY\"}\n\n\ndef regression_metrics(actual: np.ndarray, predicted: np.ndarray, *, train_direction: int = 1) -> dict:\n    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)\n    if actual.shape != predicted.shape or not actual.size or not np.isfinite(actual).all() or not np.isfinite(predicted).all():\n        raise ValueError(\"Metrics require aligned, finite, nonempty arrays\")\n    signs = np.sign(actual)\n    nonflat = signs != 0\n    classes = np.unique(signs[nonflat])\n    balanced = np.mean([np.mean(np.sign(predicted[signs == label]) == label) for label in classes]) if len(classes) else None\n    return {\"mae\": float(np.mean(np.abs(actual - predicted))), \"rmse\": float(np.sqrt(np.mean((actual - predicted) ** 2))), \"directional_accuracy\": float(np.mean(np.sign(predicted) == signs)), \"nonflat_directional_accuracy\": float(np.mean(np.sign(predicted[nonflat]) == signs[nonflat])) if nonflat.any() else None, \"balanced_directional_accuracy\": float(balanced) if balanced is not None else None, \"nonflat_train_majority_accuracy\": float(np.mean(signs[nonflat] == train_direction)) if nonflat.any() else None, \"flat_fraction\": float(np.mean(~nonflat)), \"samples\": int(actual.size)}\n\n\ndef paired_date_block_bootstrap(actual: np.ndarray, candidate: np.ndarray, reference: np.ndarray, dates: np.ndarray, *, block_sessions: int = 5, repetitions: int = 500, seed: int = 42) -> dict:\n    \"\"\"CI for reference MAE minus candidate MAE, resampling contiguous date blocks.\n\n    Tickers on a date stay together. Date sums/counts preserve row weighting;\n    overlapping horizon labels are handled by block dependence, not iid rows.\n    \"\"\"\n    if block_sessions < 1 or repetitions < 20:\n        raise ValueError(\"Invalid bootstrap configuration\")\n    # The reference and candidate must share arithmetic precision. In legacy\n    # training, float32 zero MAE and float64 candidate MAE manufactured tiny wins.\n    actual, candidate, reference = [np.asarray(value, dtype=np.float64) for value in (actual, candidate, reference)]\n    dates = np.asarray(dates)\n    if actual.ndim != 1 or any(value.shape != actual.shape for value in (candidate, reference, dates)):\n        raise ValueError(\"Bootstrap requires aligned one-dimensional arrays\")\n    if pd.isna(dates).any():\n        raise ValueError(\"Bootstrap dates cannot be missing\")\n    dates = pd.to_datetime(dates, utc=True).tz_localize(None).normalize().to_numpy()\n    frame = pd.DataFrame({\"date\": dates, \"delta\": np.abs(actual - reference) - np.abs(actual - candidate)})\n    if frame.empty or not np.isfinite(frame.delta).all():\n        raise ValueError(\"Invalid bootstrap observations\")\n    grouped = frame.groupby(\"date\", sort=True).delta.agg([\"sum\", \"count\"])\n    sums, counts = grouped[\"sum\"].to_numpy(), grouped[\"count\"].to_numpy()\n    if len(sums) < block_sessions * 2:\n        return {\"status\": \"INSUFFICIENT_DATES\", \"date_count\": len(sums), \"lower\": None, \"upper\": None}\n    rng = np.random.default_rng(seed)\n    samples = []\n    for _ in range(repetitions):\n        starts = rng.integers(0, len(sums) - block_sessions + 1, size=int(np.ceil(len(sums) / block_sessions)))\n        indices = (starts[:, None] + np.arange(block_sessions)[None, :]).reshape(-1)[:len(sums)]\n        samples.append(float(sums[indices].sum() / counts[indices].sum()))\n    return {\"status\": \"ESTIMATED\", \"estimate\": float(sums.sum() / counts.sum()), \"lower\": float(np.quantile(samples, 0.025)), \"upper\": float(np.quantile(samples, 0.975)), \"date_count\": len(sums), \"block_sessions\": block_sessions, \"repetitions\": repetitions}\n","stock_multi_agent/services/ml_model_registry.py":"\"\"\"Fail-closed artifact validation and separate research/policy gates for V3.\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom datetime import datetime\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nfrom stock_multi_agent.services.ml_dataset_v3 import sha256_file\nfrom stock_multi_agent.services.ml_features_v3 import FEATURE_VERSION, FEATURES\n\n\n# Numerical equivalence only, NOT a tunable economic improvement threshold.\n# Covers float32 aggregation versus float64 metrics in legacy V3 reports.\nGATE_IMPLEMENTATION_VERSION = \"v3-gate-numeric-fix-1\"\nMAE_ABSOLUTE_TOLERANCE = 1e-10\nMAE_RELATIVE_TOLERANCE = 1e-6\n\n\ndef _finite_number(value: object) -> bool:\n    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)\n\n\ndef mae_strictly_better(candidate: float, reference: float) -> bool:\n    \"\"\"A tie at floating-point precision is not evidence of improvement.\"\"\"\n    if not _finite_number(candidate) or not _finite_number(reference) or candidate < 0 or reference <= 0:\n        return False\n    tolerance = max(MAE_ABSOLUTE_TOLERANCE, MAE_RELATIVE_TOLERANCE * max(abs(candidate), abs(reference)))\n    return reference - candidate > tolerance\n\n\ndef research_gate_diagnostics(report: dict, protocol: dict, *, horizon: int = 1) -> dict:\n    limits = protocol[\"gate\"]\n    folds = [item for item in report.get(\"folds\", []) if item.get(\"horizon\") == horizon]\n    fold_ids = {item.get(\"fold_id\") for item in folds}\n    registered_seeds = set(protocol.get(\"seeds\", []))\n    baseline_ok = True\n    summaries = []\n    for fold_id in sorted(fold_ids, key=str):\n        members = [item for item in folds if item.get(\"fold_id\") == fold_id]\n        seeds_ok = {item.get(\"seed\") for item in members} == registered_seeds and len(members) == len(registered_seeds)\n        baseline_ok &= seeds_ok\n        member_summaries = []\n        for item in members:\n            valid = _finite_number(item.get(\"mae\")) and item[\"mae\"] >= 0\n            valid &= _finite_number(item.get(\"zero_mae\")) and item[\"zero_mae\"] > 0\n            signal = _finite_number(item.get(\"calibration_alpha\")) and 0 < item[\"calibration_alpha\"] <= 1 and item.get(\"uses_lstm_signal\") is not False\n            better = valid and mae_strictly_better(item[\"mae\"], item[\"zero_mae\"])\n            member_summaries.append({\"seed\": item.get(\"seed\"), \"strict_mae_win\": bool(better), \"uses_lstm_signal\": bool(signal)})\n            baseline_ok &= valid\n            for baseline in (\"ridge_mae\", \"tree_mae\"):\n                value = item.get(baseline)\n                if not _finite_number(value) or value <= 0 or not valid or item[\"mae\"] > value * (1 + limits[\"max_relative_loss_vs_ridge_tree\"]):\n                    baseline_ok = False\n        # A seed is a repeated trial, not a new fold. A constant calibration\n        # baseline cannot be counted as a winning LSTM even if its bias helps.\n        wins = seeds_ok and all(row[\"strict_mae_win\"] and row[\"uses_lstm_signal\"] for row in member_summaries)\n        summaries.append({\"fold_id\": fold_id, \"winning_fold\": bool(wins), \"members\": member_summaries})\n    winning_folds = sum(row[\"winning_fold\"] for row in summaries)\n    passed = (fold_ids == set(range(protocol.get(\"outer_folds\", 3)))\n        and len(fold_ids) >= limits[\"minimum_outer_folds\"]\n        and winning_folds >= limits[\"minimum_winning_folds\"] and baseline_ok)\n    return {\"gate_implementation_version\": GATE_IMPLEMENTATION_VERSION, \"horizon\": horizon,\n        \"winning_folds\": winning_folds, \"required_winning_folds\": limits[\"minimum_winning_folds\"],\n        \"ridge_tree_and_seed_checks_passed\": bool(baseline_ok), \"passed\": bool(passed), \"folds\": summaries,\n        \"numerical_tolerance\": {\"absolute\": MAE_ABSOLUTE_TOLERANCE, \"relative\": MAE_RELATIVE_TOLERANCE}}\n\n\n@dataclass(frozen=True)\nclass ModelGate:\n    artifact_status: str\n    validation_status: str\n    policy_eligibility: bool\n    technical_blend_weight: float\n    reasons: tuple[str, ...]\n\n    def as_dict(self) -> dict:\n        return {\"artifact_status\": self.artifact_status, \"validation_status\": self.validation_status, \"policy_eligibility\": self.policy_eligibility, \"technical_blend_weight\": self.technical_blend_weight, \"reasons\": list(self.reasons)}\n\n\ndef evaluate_model_gate(report: dict, protocol: dict, *, horizon: int = 1) -> ModelGate:\n    reasons: list[str] = []\n    limits = protocol[\"gate\"]\n    if report.get(\"pilot\") is True:\n        return ModelGate(\"VERIFIED_FILES\", \"PILOT_ONLY\", False, 0.0, (\"Pilot runs cannot be promoted\",))\n    finite = _finite_number\n\n    if report.get(\"protocol_id\") != protocol.get(\"protocol_id\") or horizon not in protocol.get(\"horizons\", []):\n        return ModelGate(\"VERIFIED_FILES\", \"RESEARCH_GATE_PENDING_OR_REJECTED\", False, 0.0, (\"Protocol identity/horizon mismatch\",))\n    folds = [item for item in report.get(\"folds\", []) if item.get(\"horizon\") == horizon]\n    fold_ids = {item.get(\"fold_id\") for item in folds}\n    diagnostics = research_gate_diagnostics(report, protocol, horizon=horizon)\n    if not diagnostics[\"passed\"]:\n        reasons.append(\"Development walk-forward has not cleared all pre-registered fold and Ridge/tree gates \"\n            f\"({diagnostics['winning_folds']}/{diagnostics['required_winning_folds']} required winning folds; numerical ties and alpha=0 are not wins)\")\n    # Report all known blockers, not merely the first one encountered.\n    if report.get(\"engineering_checks\") != {\"feature_parity\": True, \"causality\": True, \"purged_splits\": True, \"train_only_scaling\": True}:\n        reasons.append(\"Engineering checks are missing or not all verified\")\n    if limits.get(\"require_adjusted_prices_for_blend\") and report.get(\"price_adjustment\") != \"ADJUSTED\":\n        reasons.append(\"Verified adjusted prices are required for Technical blend\")\n    latest = [item for item in folds if item.get(\"fold_id\") == protocol.get(\"outer_folds\", 3) - 1]\n    if not any(finite(item.get(\"calibration_alpha\")) and item[\"calibration_alpha\"] > 0 for item in latest):\n        reasons.append(\"Latest-fold model has no calibrated LSTM signal; constant baselines cannot receive LSTM blend weight\")\n    forward = report.get(\"forward\", {}).get(str(horizon), {})\n    independent = forward.get(\"independent\") is True and forward.get(\"protocol_id\") == protocol.get(\"protocol_id\")\n    try:\n        frozen = datetime.fromisoformat(protocol[\"frozen_at\"].replace(\"Z\", \"+00:00\"))\n        captured = datetime.fromisoformat(forward.get(\"capture_started_at\", \"\").replace(\"Z\", \"+00:00\"))\n        independent &= frozen.utcoffset() is not None and captured.utcoffset() is not None and captured >= frozen\n    except (ValueError, TypeError, KeyError):\n        independent = False\n    sessions = forward.get(\"sessions\", 0)\n    forward_pending = not independent or not isinstance(sessions, int) or isinstance(sessions, bool) or sessions < limits[\"minimum_forward_sessions\"]\n    if forward_pending:\n        reasons.append(\"A new, protocol-locked independent forward holdout with enough observed sessions is required\")\n    else:\n        for key in (\"mae_improvement_ci\", \"direction_improvement_ci\"):\n            interval = forward.get(key, {})\n            if interval.get(\"status\") != \"ESTIMATED\" or not finite(interval.get(\"lower\")) or interval[\"lower\"] <= 0:\n                reasons.append(f\"{key} is inconclusive or does not improve the registered baseline\")\n    if not diagnostics[\"passed\"]:\n        return ModelGate(\"VERIFIED_FILES\", \"RESEARCH_GATE_PENDING_OR_REJECTED\", False, 0.0, tuple(reasons))\n    if forward_pending:\n        return ModelGate(\"VERIFIED_FILES\", \"PENDING_FORWARD_VALIDATION\", False, 0.0, tuple(reasons))\n    if reasons:\n        return ModelGate(\"VERIFIED_FILES\", \"INCONCLUSIVE\", False, 0.0, tuple(reasons))\n    return ModelGate(\"VERIFIED_FILES\", \"FORWARD_VALIDATED\", True, min(0.2, limits[\"maximum_technical_weight\"]), ())\n\n\ndef verify_artifact_directory(directory: Path) -> tuple[dict, dict, dict, dict]:\n    \"\"\"No arbitrary manifest paths or custom Keras objects; all files are pinned.\"\"\"\n    directory = directory.resolve()\n    manifest = json.loads((directory / \"manifest.json\").read_text(encoding=\"utf-8\"))\n    if manifest.get(\"schema_version\") != \"lstm-artifact-v3\":\n        raise ValueError(\"Unsupported V3 manifest schema\")\n    required = {\"model.keras\", \"spec.json\", \"evaluation.json\", \"protocol.json\"}\n    files = manifest.get(\"files\", {})\n    if set(files) != required:\n        raise ValueError(\"Manifest must pin exactly the V3 model/spec/evaluation/protocol files\")\n    for filename in required:\n        path = directory / filename\n        if not path.is_file() or path.is_symlink() or path.resolve().parent != directory or sha256_file(path) != files[filename]:\n            raise ValueError(f\"V3 artifact hash/path mismatch: {filename}\")\n    spec = json.loads((directory / \"spec.json\").read_text(encoding=\"utf-8\"))\n    report = json.loads((directory / \"evaluation.json\").read_text(encoding=\"utf-8\"))\n    protocol = json.loads((directory / \"protocol.json\").read_text(encoding=\"utf-8\"))\n    if spec.get(\"feature_version\") != FEATURE_VERSION or spec.get(\"features\") != FEATURES:\n        raise ValueError(\"V3 shared feature version/order mismatch\")\n    if spec.get(\"horizons\") != [1, 5] or spec.get(\"window\") != protocol.get(\"window\") or spec.get(\"window\") != 30:\n        raise ValueError(\"V3 horizon/window mismatch\")\n    if spec.get(\"protocol_id\") != protocol.get(\"protocol_id\") or report.get(\"protocol_id\") != protocol.get(\"protocol_id\"):\n        raise ValueError(\"Protocol identity mismatch\")\n    expected_identity = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()\n    if manifest.get(\"artifact_identity\") != expected_identity:\n        raise ValueError(\"Artifact identity does not match its pinned files\")\n    predictor_identity = hashlib.sha256(json.dumps({name: files[name] for name in (\"model.keras\", \"spec.json\", \"protocol.json\")}, sort_keys=True).encode()).hexdigest()\n    if manifest.get(\"predictor_identity\") != predictor_identity:\n        raise ValueError(\"Predictor identity does not match model/spec/protocol\")\n    for forward in report.get(\"forward\", {}).values():\n        if forward.get(\"predictor_identity\") != predictor_identity:\n            raise ValueError(\"Forward evidence belongs to a different frozen predictor\")\n    return manifest, spec, report, protocol\n\n\ndef write_artifact_manifest(directory: Path, *, code_revision: str = \"UNRECORDED\") -> dict:\n    files = {name: sha256_file(directory / name) for name in (\"model.keras\", \"spec.json\", \"evaluation.json\", \"protocol.json\")}\n    predictor_identity = hashlib.sha256(json.dumps({name: files[name] for name in (\"model.keras\", \"spec.json\", \"protocol.json\")}, sort_keys=True).encode()).hexdigest()\n    manifest = {\"schema_version\": \"lstm-artifact-v3\", \"files\": files, \"code_revision\": code_revision, \"artifact_identity\": hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest(), \"predictor_identity\": predictor_identity, \"identity_policy\": \"predictor_identity excludes evaluation; artifact_identity pins the complete reviewed release\", \"default_policy\": \"SHADOW_UNTIL_FORWARD_GATE\"}\n    (directory / \"manifest.json\").write_text(json.dumps(manifest, indent=2), encoding=\"utf-8\")\n    return manifest\n","stock_multi_agent/services/ml_dataset_v3.py":"\"\"\"Auditable V3 dataset builder. Original data and V2 are never modified.\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\n\nfrom stock_multi_agent.services.ml_features_v3 import FEATURE_VERSION, FEATURES, add_targets, build_feature_frame, window_endpoints\n\n\n@dataclass\nclass DatasetBuildResult:\n    rows: pd.DataFrame\n    quarantine: pd.DataFrame\n    manifest: dict\n\n\nRUNTIME_MODULES = (\"ml_features_v3.py\", \"ml_dataset_v3.py\", \"ml_validation.py\", \"ml_baselines.py\", \"ml_model_registry.py\")\n\n\ndef _write_runtime_bundle(bundle: Path) -> None:\n    service_root = Path(__file__).resolve().parent\n    with zipfile.ZipFile(bundle, \"x\", compression=zipfile.ZIP_DEFLATED) as archive:\n        archive.writestr(\"stock_multi_agent/__init__.py\", '\"\"\"Portable V3 training runtime.\"\"\"\\n')\n        archive.writestr(\"stock_multi_agent/services/__init__.py\", '\"\"\"Portable causal ML modules.\"\"\"\\n')\n        for filename in RUNTIME_MODULES:\n            archive.write(service_root / filename, f\"stock_multi_agent/services/{filename}\")\n\n\ndef refresh_runtime_bundle(output_dir: Path) -> dict:\n    \"\"\"Explicit engineering refresh, preserving CSVs and the prior runtime ZIP.\n\n    No training or performance result is rewritten. A content-addressed new\n    runtime bundle and history entry make the preparation change inspectable.\n    \"\"\"\n    manifest_path = output_dir / \"dataset.manifest.json\"\n    manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    dataset_name = manifest[\"files\"][\"dataset\"][\"filename\"]\n    if Path(dataset_name).name != dataset_name or sha256_file(output_dir / dataset_name) != manifest[\"files\"][\"dataset\"][\"sha256\"]:\n        raise ValueError(\"Cannot refresh a changed dataset or unsafe dataset filename\")\n    service_root = Path(__file__).resolve().parent\n    code_hash = hashlib.sha256(\"\".join(sha256_file(service_root / name) for name in RUNTIME_MODULES).encode()).hexdigest()\n    bundle = output_dir / f\"ml_v3_runtime.{code_hash[:12]}.zip\"\n    if manifest[\"runtime_bundle\"].get(\"filename\") == bundle.name:\n        if sha256_file(bundle) != manifest[\"runtime_bundle\"][\"sha256\"]:\n            raise ValueError(\"Current runtime bundle was modified\")\n        return manifest\n    _write_runtime_bundle(bundle)\n    manifest.setdefault(\"runtime_bundle_history\", []).append(dict(manifest[\"runtime_bundle\"]))\n    manifest[\"runtime_bundle\"] = {\"filename\": bundle.name, \"sha256\": sha256_file(bundle), \"code_sha256\": code_hash}\n    manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding=\"utf-8\")\n    return manifest\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\"\"):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef build_dataset(raw: pd.DataFrame, *, window: int = 30, max_gap_business_days: int = 5) -> DatasetBuildResult:\n    required = {\"ticker\", \"date\", \"open\", \"high\", \"low\", \"close\", \"volume\", \"source\", \"source_id\", \"fetched_at\", \"price_adjustment\"}\n    missing = required.difference(raw.columns)\n    if missing:\n        raise ValueError(f\"Missing source data columns: {sorted(missing)}\")\n    frame = raw.copy()\n    frame[\"input_row\"] = np.arange(len(frame))\n    frame[\"ticker\"] = frame[\"ticker\"].astype(str).str.upper().str.strip()\n    frame[\"date\"] = pd.to_datetime(frame[\"date\"], errors=\"coerce\", utc=True).dt.tz_localize(None).dt.normalize()\n    for name in (\"open\", \"high\", \"low\", \"close\", \"volume\"):\n        frame[name] = pd.to_numeric(frame[name], errors=\"coerce\")\n    frame[\"available_at\"] = pd.to_datetime(frame[\"fetched_at\"], errors=\"coerce\", utc=True)\n    frame[\"price_unit\"] = frame.get(\"price_unit\", \"UNKNOWN\")\n    frame[\"price_unit\"] = frame[\"price_unit\"].fillna(\"UNKNOWN\")\n    frame[\"price_adjustment\"] = frame[\"price_adjustment\"].fillna(\"UNKNOWN\").astype(str).str.upper()\n    reason = pd.Series(\"\", index=frame.index)\n    prices = frame[[\"open\", \"high\", \"low\", \"close\"]]\n    bad = ~np.isfinite(frame[[\"open\", \"high\", \"low\", \"close\", \"volume\"]]).all(axis=1)\n    bad |= prices.le(0).any(axis=1) | frame[\"volume\"].lt(0)\n    bad |= frame[\"high\"].lt(prices[[\"open\", \"low\", \"close\"]].max(axis=1))\n    bad |= frame[\"low\"].gt(prices[[\"open\", \"high\", \"close\"]].min(axis=1))\n    reason.loc[bad] = \"INVALID_OHLCV\"\n    bad_metadata = frame[\"date\"].isna() | frame[\"available_at\"].isna() | ~frame[\"ticker\"].str.fullmatch(r\"[A-Z][A-Z0-9]{1,9}\")\n    for column in (\"source\", \"source_id\"):\n        bad_metadata |= frame[column].isna() | frame[column].astype(str).str.strip().eq(\"\")\n    bad_metadata |= frame[\"date\"].gt(frame[\"available_at\"].dt.tz_localize(None).dt.normalize())\n    reason.loc[bad_metadata] = \"INVALID_DATE_OR_PROVENANCE\"\n    # Identical duplicates are explicitly reconciled; conflicting bars all quarantine.\n    identity = [\"ticker\", \"date\"]\n    exact = frame.duplicated(identity + [\"open\", \"high\", \"low\", \"close\", \"volume\", \"source_id\", \"price_adjustment\", \"price_unit\"], keep=\"first\")\n    reason.loc[exact & reason.eq(\"\")] = \"EXACT_DUPLICATE\"\n    candidates = frame[reason.eq(\"\")]\n    conflict_ids = candidates.index[candidates.duplicated(identity, keep=False)]\n    reason.loc[conflict_ids] = \"CONFLICTING_DUPLICATE\"\n    frame[\"quarantine_reason\"] = reason\n    quarantine = frame[reason.ne(\"\")].copy()\n    clean = frame[reason.eq(\"\")].copy().sort_values([\"ticker\", \"date\"]).reset_index(drop=True)\n    if clean.empty:\n        raise ValueError(\"No valid source rows remain\")\n    # A removed bad bar must break the next segment even if the calendar gap is short.\n    bad_dates = quarantine[~quarantine[\"quarantine_reason\"].eq(\"EXACT_DUPLICATE\")].dropna(subset=[\"date\"])\n    breaks = clean[\"ticker\"].ne(clean[\"ticker\"].shift())\n    previous = clean.groupby(\"ticker\")[\"date\"].shift(1)\n    good_previous = previous.notna()\n    gaps = np.zeros(len(clean), dtype=np.int64)\n    gaps[good_previous] = np.busday_count(previous[good_previous].to_numpy(dtype=\"datetime64[D]\"), clean.loc[good_previous, \"date\"].to_numpy(dtype=\"datetime64[D]\"))\n    breaks |= gaps > max_gap_business_days\n    for _, indices in clean.groupby(\"ticker\").groups.items():\n        ticker = clean.loc[indices[0], \"ticker\"]\n        removed = bad_dates.loc[bad_dates[\"ticker\"].eq(ticker), \"date\"].to_numpy(dtype=\"datetime64[ns]\")\n        if len(removed):\n            dates = clean.loc[indices, \"date\"].to_numpy(dtype=\"datetime64[ns]\")\n            removed = np.sort(removed)\n            positions = np.searchsorted(dates, removed, side=\"right\")\n            positions = positions[positions < len(indices)]\n            breaks.loc[np.asarray(indices)[positions]] = True\n    # Mixing provider scale/adjustment regimes inside a window is not allowed.\n    for column in (\"price_unit\", \"price_adjustment\", \"source\"):\n        breaks |= clean[column].ne(clean.groupby(\"ticker\")[column].shift())\n    clean[\"segment_id\"] = breaks.cumsum().astype(int)\n    clean = add_targets(build_feature_frame(clean, max_gap_business_days=max_gap_business_days))\n    endpoints = window_endpoints(clean, window)\n    clean[\"training_endpoint\"] = False\n    clean.loc[endpoints, \"training_endpoint\"] = True\n    manifest = {\n        \"schema_version\": \"lstm-dataset-v3\", \"feature_version\": FEATURE_VERSION,\n        \"created_at\": datetime.now(timezone.utc).isoformat(), \"window\": window, \"features\": FEATURES,\n        \"input_rows\": len(raw), \"retained_rows\": len(clean), \"quarantined_rows\": len(quarantine),\n        \"reconciled_counts\": len(clean) + len(quarantine) == len(raw),\n        \"quarantine_reasons\": quarantine[\"quarantine_reason\"].value_counts().to_dict(),\n        \"training_endpoints\": len(endpoints), \"tickers\": int(clean.ticker.nunique()),\n        \"segments\": int(clean.segment_id.nunique()), \"max_gap_business_days\": max_gap_business_days,\n        \"label_policy\": \"Observed bars within a segment; no future-return magnitude filtering\",\n        \"known_at_policy\": \"available_at is actual fetched_at; historical forecast_origin is simulated close, not point-in-time availability\",\n        \"calendar_policy\": \"Observed source bars; conservative weekday gap segmentation, not certified exchange calendar\",\n        \"adjustment_counts\": clean.price_adjustment.value_counts().to_dict(),\n        \"price_unit_counts\": clean.price_unit.value_counts().to_dict(),\n        \"universe_policy\": \"Source current-universe snapshot; survivorship bias not resolved\",\n        \"validation_status\": \"DEVELOPMENT_ONLY\", \"policy_eligibility\": \"SHADOW_ONLY\",\n    }\n    return DatasetBuildResult(clean, quarantine, manifest)\n\n\ndef write_dataset(result: DatasetBuildResult, output_dir: Path, *, source_path: Path | None = None) -> dict:\n    # Refuse a partially populated directory too: a failed earlier export must\n    # not be mixed with fresh CSVs or a stale shared-code ZIP.\n    if output_dir.exists() and any(output_dir.iterdir()):\n        raise FileExistsError(\"V3 output directory is not empty; choose a new --output directory\")\n    output_dir.mkdir(parents=True, exist_ok=True)\n    paths = {\"dataset\": output_dir / \"vci_ohlcv_lstm_v3.csv\", \"quarantine\": output_dir / \"quarantine.csv\", \"manifest\": output_dir / \"dataset.manifest.json\"}\n    if any(path.exists() for path in paths.values()):\n        raise FileExistsError(\"V3 output exists; choose a new --output directory to preserve the experiment\")\n    result.rows.to_csv(paths[\"dataset\"], index=False)\n    result.quarantine.to_csv(paths[\"quarantine\"], index=False)\n    manifest = dict(result.manifest)\n    manifest[\"files\"] = {key: {\"filename\": path.name, \"sha256\": sha256_file(path)} for key, path in paths.items() if key != \"manifest\"}\n    if source_path:\n        manifest[\"source_file\"] = {\"filename\": source_path.name, \"sha256\": sha256_file(source_path)}\n    # Portable shared code, without importing the full app/dependencies on Kaggle.\n    # The upload is private/user-controlled; raw reports and credentials are absent.\n    bundle = output_dir / \"ml_v3_runtime.zip\"\n    _write_runtime_bundle(bundle)\n    manifest[\"runtime_bundle\"] = {\"filename\": bundle.name, \"sha256\": sha256_file(bundle)}\n    paths[\"manifest\"].write_text(json.dumps(manifest, indent=2, default=str), encoding=\"utf-8\")\n    return manifest\n","stock_multi_agent/services/ml_baselines.py":"\"\"\"Free baselines evaluated on the exact same endpoints as the LSTM.\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom stock_multi_agent.services.ml_features_v3 import FEATURES\nfrom stock_multi_agent.services.ml_validation import TrainOnlyScaler, WalkForwardFold, regression_metrics\n\n\ndef baseline_predictions(frame: pd.DataFrame, fold: WalkForwardFold, horizon: int, *, include_tree: bool = True) -> tuple[dict[str, np.ndarray], dict]:\n    target = f\"target_return_{horizon}\"\n    train = frame.iloc[fold.train]\n    y_train = train[target].to_numpy(dtype=float)\n    y_val = frame.iloc[fold.validation][target].to_numpy(dtype=float)\n    scaler = TrainOnlyScaler().fit(train[FEATURES].to_numpy(float))\n    x_train = scaler.transform(train[FEATURES].to_numpy(float)).astype(float)\n    x_val = scaler.transform(frame.iloc[fold.validation][FEATURES].to_numpy(float)).astype(float)\n    x_test = scaler.transform(frame.iloc[fold.test][FEATURES].to_numpy(float)).astype(float)\n    n = len(fold.test)\n    predictions = {\"zero\": np.zeros(n), \"train_mean\": np.full(n, y_train.mean()), \"train_median\": np.full(n, np.median(y_train)), \"persistence\": frame.iloc[fold.test][f\"return_{horizon}\"].to_numpy(float)}\n    # Ridge hyperparameter selection is validation-only, never outer test.\n    design = np.column_stack([np.ones(len(x_train)), x_train])\n    val_design = np.column_stack([np.ones(len(x_val)), x_val])\n    best = None\n    for alpha in (1.0, 100.0, 1000.0):\n        penalty = np.eye(design.shape[1]) * alpha\n        penalty[0, 0] = 0\n        coefficient = np.linalg.solve(design.T @ design + penalty, design.T @ y_train)\n        val_mae = float(np.mean(np.abs(y_val - val_design @ coefficient)))\n        if best is None or val_mae < best[0]:\n            best = (val_mae, alpha, coefficient)\n    predictions[\"ridge\"] = np.column_stack([np.ones(len(x_test)), x_test]) @ best[2]\n    details = {\"ridge_alpha\": best[1], \"ridge_validation_mae\": best[0], \"tree_status\": \"DISABLED\", \"fit_scope\": \"TRAIN_ONLY\", \"selected_using\": \"VALIDATION_ONLY\"}\n    if include_tree:\n        try:\n            from sklearn.ensemble import HistGradientBoostingRegressor\n        except ImportError:\n            details[\"tree_status\"] = \"UNAVAILABLE_OPTIONAL_SKLEARN\"\n        else:\n            tree = HistGradientBoostingRegressor(max_iter=60, max_leaf_nodes=7, l2_regularization=10, learning_rate=0.05, early_stopping=False, random_state=42)\n            try:\n                tree.fit(x_train, y_train)\n                predictions[\"tree\"] = tree.predict(x_test)\n                details[\"tree_status\"] = \"EVALUATED\"\n            except (OSError, RuntimeError) as exc:\n                # Resource-limited classroom environments may deny worker\n                # pipes/threads. Keep other baseline results; missing tree\n                # remains explicit and MUST fail the promotion gate.\n                details[\"tree_status\"] = \"EXECUTION_UNAVAILABLE\"\n                details[\"tree_error_type\"] = type(exc).__name__\n    return predictions, details\n\n\ndef evaluate_baselines(frame: pd.DataFrame, folds: list[WalkForwardFold], *, include_tree: bool = True) -> dict:\n    reports = []\n    for fold in folds:\n        for horizon in (1, 5):\n            predictions, details = baseline_predictions(frame, fold, horizon, include_tree=include_tree)\n            actual = frame.iloc[fold.test][f\"target_return_{horizon}\"].to_numpy(float)\n            train_signs = np.sign(frame.iloc[fold.train][f\"target_return_{horizon}\"].to_numpy(float))\n            train_sign = 1 if np.count_nonzero(train_signs > 0) >= np.count_nonzero(train_signs < 0) else -1\n            reports.append({\"fold_id\": fold.fold_id, \"horizon\": horizon, \"metrics\": {name: regression_metrics(actual, predicted, train_direction=train_sign) for name, predicted in predictions.items()}, \"details\": details, \"split\": fold.manifest()})\n    return {\"schema_version\": \"ml-baselines-v3\", \"evaluation_scope\": \"DEVELOPMENT_WALK_FORWARD\", \"same_endpoints\": True, \"reports\": reports}\n","stock_multi_agent/services/ml_v4.py":"\"\"\"V4 causal features, eligible-session splits and flat-aware calibration.\n\nNo TensorFlow import, downloads, model loading or training at import time.\nV3 source data/targets and V2 artifacts are never modified.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom stock_multi_agent.services.ml_features_v3 import FEATURES as V3_FEATURES, build_feature_frame, window_endpoints\nfrom stock_multi_agent.services.ml_validation import WalkForwardFold, TrainOnlyScaler, regression_metrics\n\nFEATURE_VERSION = 'single-ticker-v4-1'\nFEATURES = [*V3_FEATURES, 'flat_ratio_5', 'flat_ratio_20', 'volume_cv_20', 'turnover_ratio_20',\n    'candle_body_return', 'upper_wick_ratio', 'lower_wick_ratio', 'return_mean_5']\n\n\ndef feature_frame(raw: pd.DataFrame, *, reuse_verified_v3_features=False) -> pd.DataFrame:\n    if reuse_verified_v3_features:\n        # Only the trainer may opt in after hashing the frozen V3 CSV. Runtime\n        # builds from raw bars by default; tests prove both paths are identical.\n        required = set(V3_FEATURES) | {'ticker', 'date', 'segment_id', 'open', 'high', 'low', 'close', 'volume'}\n        if required.difference(raw.columns):\n            raise ValueError('Verified V3 cache is missing feature/source columns')\n        frame = raw.copy()\n        frame['date'] = pd.to_datetime(frame.date).dt.tz_localize(None).dt.normalize()\n        frame = frame.sort_values(['ticker', 'date']).reset_index(drop=True)\n        if frame.duplicated(['ticker', 'date']).any():\n            raise ValueError('Duplicate ticker/date in V3 cache')\n    else:\n        frame = build_feature_frame(raw)\n    keys = [frame.ticker, frame.segment_id]\n    previous_close = frame.groupby(['ticker', 'segment_id'], sort=False).close.shift(1)\n    flat = frame.return_1.abs().le(1e-12).astype(float).where(frame.return_1.notna())\n    for period in (5, 20):\n        frame[f'flat_ratio_{period}'] = flat.groupby(keys).transform(lambda x: x.rolling(period, min_periods=period).mean())\n    grouped = frame.groupby(['ticker', 'segment_id'], sort=False)\n    vol_mean = grouped.volume.transform(lambda x: x.rolling(20, min_periods=20).mean())\n    vol_std = grouped.volume.transform(lambda x: x.rolling(20, min_periods=20).std(ddof=1))\n    frame['volume_cv_20'] = vol_std / vol_mean.replace(0, np.nan)\n    turnover = frame.close * frame.volume\n    frame['turnover_ratio_20'] = turnover / turnover.groupby(keys).transform(lambda x: x.rolling(20, min_periods=20).mean()).replace(0, np.nan)\n    frame['candle_body_return'] = (frame.close - frame.open) / previous_close\n    spread = frame.high - frame.low\n    frame['upper_wick_ratio'] = ((frame.high - frame[['open', 'close']].max(axis=1)) / spread.replace(0, np.nan)).where(spread.ne(0), 0.)\n    frame['lower_wick_ratio'] = ((frame[['open', 'close']].min(axis=1) - frame.low) / spread.replace(0, np.nan)).where(spread.ne(0), 0.)\n    frame['return_mean_5'] = frame.return_1.groupby(keys).transform(lambda x: x.rolling(5, min_periods=5).mean())\n    finite = np.isfinite(frame[FEATURES].to_numpy(dtype=float)).all(axis=1)\n    frame['feature_valid'] = finite\n    frame['endpoint_eligible'] = finite & frame.volume.gt(0) & frame.active_ratio_20.ge(.5)\n    return frame\n\n\ndef eligible_session_folds(frame, endpoints, *, folds=3, gap_sessions=5, minimum_sessions=None):\n    \"\"\"Allocate by actual eligible dates; still count gap on full source calendar.\n\n    No future-return value is used for boundaries. Availability of target dates is\n    required for purging only. Sparse validation never silently passes as months.\n    \"\"\"\n    minimum_sessions = minimum_sessions or {'train': 120, 'validation': 30, 'calibration': 30, 'test': 40}\n    if folds < 1 or gap_sessions < 0:\n        raise ValueError('Invalid folds/gap')\n    endpoints = np.asarray(endpoints, dtype=np.int64)\n    if not len(endpoints):\n        raise ValueError('No eligible endpoints')\n    origins = pd.to_datetime(frame.iloc[endpoints].date).to_numpy(dtype='datetime64[ns]')\n    labels = pd.to_datetime(frame.iloc[endpoints].target_end_5).to_numpy(dtype='datetime64[ns]')\n    eligible_dates = np.unique(origins)\n    calendar = np.sort(pd.to_datetime(frame.date).dropna().unique().to_numpy(dtype='datetime64[ns]'))\n    initial = int(len(eligible_dates) * .45)\n    block = (len(eligible_dates) - initial) // (folds * 2)\n    if block < 4:\n        raise ValueError('Too few eligible dates; collect more history')\n    def after_gap(value):\n        index = np.searchsorted(calendar, value) + gap_sessions + 1\n        if index >= len(calendar):\n            raise ValueError('Not enough source sessions for gap')\n        return calendar[index]\n    result = []\n    for number in range(folds):\n        train_end = eligible_dates[initial + number * 2 * block - 1]\n        val_end = eligible_dates[initial + (number * 2 + 1) * block - 1]\n        selection_end = eligible_dates[initial + number * 2 * block + block // 2 - 1]\n        test_end = eligible_dates[-1] if number == folds - 1 else eligible_dates[initial + (number * 2 + 2) * block - 1]\n        val_start, cal_start, test_start = after_gap(train_end), after_gap(selection_end), after_gap(val_end)\n        masks = {'train': (origins <= train_end) & (labels <= train_end),\n            'validation': (origins >= val_start) & (labels <= selection_end),\n            'calibration': (origins >= cal_start) & (labels <= val_end),\n            'test': (origins >= test_start) & (origins <= test_end) & (labels <= test_end)}\n        indices = {name: endpoints[mask] for name, mask in masks.items()}\n        counts = {name: int(frame.iloc[value].date.nunique()) for name, value in indices.items()}\n        if any(counts[name] < minimum_sessions[name] for name in indices):\n            raise ValueError(f'Fold {number} has insufficient eligible sessions: {counts}; minimum {minimum_sessions}. Do not shrink the gate after test results.')\n        boundaries = {'train_end': str(pd.Timestamp(train_end).date()), 'validation_start': str(pd.Timestamp(val_start).date()),\n            'validation_end': str(pd.Timestamp(selection_end).date()), 'calibration_start': str(pd.Timestamp(cal_start).date()),\n            'calibration_end': str(pd.Timestamp(val_end).date()), 'test_start': str(pd.Timestamp(test_start).date()),\n            'test_end': str(pd.Timestamp(test_end).date()), 'gap_sessions': str(gap_sessions),\n            'eligible_session_counts': counts, 'boundary_policy': 'ELIGIBLE_ORIGIN_DATES_GAP_ON_SOURCE_CALENDAR'}\n        result.append(WalkForwardFold(number, **indices, boundaries=boundaries))\n    return result\n\n\ndef origin_scales(frame, floor=.005, ceiling=.10):\n    \"\"\"Known-at-origin volatility only; no target or per-ticker future statistic.\"\"\"\n    vol = frame.volatility_20.to_numpy(dtype=np.float64)\n    # Warm-up rows are never endpoints. Fixed floor/ceiling belong to protocol.\n    vol = np.clip(np.nan_to_num(vol, nan=floor), floor, ceiling)\n    return np.column_stack([vol, vol * np.sqrt(5.)])\n\n\ndef direction_labels(actual):\n    values = np.asarray(actual, dtype=np.float64)\n    if not np.isfinite(values).all():\n        raise ValueError('Direction labels require finite targets')\n    return np.where(values < -1e-12, 0, np.where(values > 1e-12, 2, 1)).astype(np.int32)\n\n\ndef direction_class_weights(train_labels):\n    labels = np.asarray(train_labels)\n    if not len(labels) or not np.isin(labels, [0, 1, 2]).all():\n        raise ValueError('Invalid training direction labels')\n    counts = np.bincount(labels, minlength=3)\n    return np.clip(len(labels) / (3 * np.maximum(counts, 1)), .5, 3.).astype(np.float32)\n\n\ndef split_calibration(frame, indices):\n    indices = np.asarray(indices)\n    dates = np.sort(pd.to_datetime(frame.iloc[indices].date).unique())\n    boundary = dates[int(len(dates) * .65)]\n    rows = frame.iloc[indices]\n    fit = indices[(pd.to_datetime(rows.target_end_5) < boundary).to_numpy()]\n    interval = indices[(pd.to_datetime(rows.date) >= boundary).to_numpy()]\n    if min(frame.iloc[fit].date.nunique(), frame.iloc[interval].date.nunique()) < 10:\n        raise ValueError('Need >=10 eligible dates each for purged calibration-fit and interval blocks')\n    return fit, interval\n\n\ndef _validate_calibration_arrays(actual, raw, flat_probability):\n    arrays = [np.asarray(x, dtype=np.float64) for x in (actual, raw, flat_probability)]\n    if arrays[0].ndim != 1 or not arrays[0].size or any(x.shape != arrays[0].shape or not np.isfinite(x).all() for x in arrays):\n        raise ValueError('Calibration arrays must be finite, aligned and nonempty')\n    if ((arrays[2] < 0) | (arrays[2] > 1)).any():\n        raise ValueError('Flat probabilities must be in [0,1]')\n    return arrays\n\n\ndef apply_calibration(raw, flat_probability, specification):\n    raw, probability = [np.asarray(x, dtype=np.float64) for x in (raw, flat_probability)]\n    if raw.shape != probability.shape or not np.isfinite(raw).all() or not np.isfinite(probability).all() or ((probability < 0) | (probability > 1)).any():\n        raise ValueError('Invalid calibrated forecast inputs')\n    alpha, bias, threshold = specification['alpha'], specification['bias'], specification['flat_threshold']\n    if not np.isfinite([alpha, bias]).all() or not 0 <= alpha <= 1 or (threshold is not None and (not np.isfinite(threshold) or not 0 <= threshold <= 1)):\n        raise ValueError('Invalid calibration parameters')\n    forecast = alpha * raw + bias\n    if threshold is not None:\n        forecast = np.where(probability >= threshold, 0., forecast)\n    return forecast\n\n\ndef fit_calibration(actual, raw, flat_probability, *, allow_flat_gate=True):\n    \"\"\"Fit only on the registered calibration-fit block, never test/interval.\"\"\"\n    actual, raw, probability = _validate_calibration_arrays(actual, raw, flat_probability)\n    options = []\n    for threshold in ([None, .5, .65, .8] if allow_flat_gate else [None]):\n        keep = np.ones(len(actual), dtype=bool) if threshold is None else probability < threshold\n        if not keep.any():\n            continue\n        for alpha in np.linspace(0, 1, 11):\n            bias = float(np.median(actual[keep] - alpha * raw[keep]))\n            spec = {'alpha': float(alpha), 'bias': bias, 'flat_threshold': threshold}\n            prediction = apply_calibration(raw, probability, spec)\n            loss = float(np.mean(np.abs(actual - prediction), dtype=np.float64))\n            options.append((loss, spec))\n    loss, best = min(options, key=lambda x: x[0])\n    return {**best, 'fit_mae': loss, 'fit_scope': 'PURGED_CALIBRATION_FIT_ONLY'}\n\n\ndef classification_metrics(actual, probabilities):\n    labels = direction_labels(actual)\n    probabilities = np.asarray(probabilities, dtype=np.float64)\n    if probabilities.shape != (len(labels), 3) or not np.isfinite(probabilities).all() or (probabilities < 0).any() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):\n        raise ValueError('Invalid direction probabilities')\n    predicted = probabilities.argmax(axis=1)\n    recall = [float(np.mean(predicted[labels == label] == label)) if (labels == label).any() else None for label in range(3)]\n    return {'three_class_accuracy': float(np.mean(predicted == labels)),\n        'balanced_three_class_accuracy': float(np.mean([x for x in recall if x is not None])),\n        'per_class_recall_down_flat_up': recall,\n        'probability_status': 'UNCALIBRATED_CLASSIFIER_NOT_TRADE_CONFIDENCE'}\n\n\ndef baseline_predictions(frame, fold, horizon, *, include_tree=True):\n    \"\"\"25-feature Ridge/tree, same endpoints and original unfiltered test labels.\"\"\"\n    train = frame.iloc[fold.train]\n    scaler = TrainOnlyScaler().fit(train[FEATURES].to_numpy(float))\n    x, v, t = [scaler.transform(frame.iloc[index][FEATURES].to_numpy(float)).astype(np.float64) for index in (fold.train, fold.validation, fold.test)]\n    y, vy = [frame.iloc[index][f'target_return_{horizon}'].to_numpy(float) for index in (fold.train, fold.validation)]\n    values = {'zero': np.zeros(len(t)), 'train_mean': np.full(len(t), y.mean()),\n        'train_median': np.full(len(t), np.median(y)), 'persistence': frame.iloc[fold.test][f'return_{horizon}'].to_numpy(float)}\n    x, v, t = [np.column_stack([np.ones(len(a)), a]) for a in (x, v, t)]\n    best = None\n    for alpha in (1., 100., 1000.):\n        penalty = np.eye(x.shape[1]) * alpha\n        penalty[0, 0] = 0\n        coefficient = np.linalg.solve(x.T @ x + penalty, x.T @ y)\n        loss = np.mean(np.abs(vy - v @ coefficient))\n        if best is None or loss < best[0]:\n            best = loss, alpha, coefficient\n    values['ridge'] = t @ best[2]\n    details = {'ridge_alpha': best[1], 'tree_status': 'DISABLED', 'feature_count': len(FEATURES), 'selection_scope': 'VALIDATION_ONLY'}\n    if include_tree:\n        try:\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            model = HistGradientBoostingRegressor(max_iter=60, max_leaf_nodes=7, l2_regularization=10,\n                learning_rate=.05, early_stopping=False, random_state=42)\n            model.fit(x[:, 1:], y)\n            values['tree'] = model.predict(t[:, 1:])\n            details['tree_status'] = 'EVALUATED'\n        except (ImportError, OSError, RuntimeError) as exc:\n            details.update(tree_status='UNAVAILABLE', tree_error_type=type(exc).__name__)\n    return values, details\n\n\ndef scaler_specification(scaler):\n    spec = scaler.specification()\n    # The generic V3 scaler's mathematical implementation is reused, not its\n    # 17-feature metadata. Pin the actual 25-feature order for future inference.\n    spec['features'] = list(FEATURES)\n    return spec\n","stock_multi_agent/__init__.py":"\"\"\"Portable V4 runtime.\"\"\"\n","stock_multi_agent/services/__init__.py":"\"\"\"No app startup or model loading.\"\"\"\n"}

TRAINING_SOURCE = "\"\"\"V4 one-session-priority experiment. Import is safe: no TensorFlow/training.\n\nThe ready .ipynb embeds this script, the pinned protocol and shared runtime.\nReuse the existing V3 CSV+manifest; no extra dataset or ZIP upload is required.\nHistorical results are DEVELOPMENT, not a newly independent holdout.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\n\ndef build_model(tf, window, feature_count, config, *, variant='priority_multitask'):\n    inputs = tf.keras.Input((window, feature_count), name='market_sequence')\n    history = tf.keras.layers.LSTM(32, recurrent_dropout=0., kernel_regularizer=tf.keras.regularizers.l2(config['l2']))(inputs)\n    last = tf.keras.layers.Flatten()(tf.keras.layers.Cropping1D((window - 1, 0))(inputs))\n    last = tf.keras.layers.Dense(16, activation='relu')(last)\n    combined = tf.keras.layers.Concatenate()([history, last])\n    combined = tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(config['l2']))(combined)\n    combined = tf.keras.layers.Dropout(config['dropout'])(combined)\n    outputs = {'return_1': tf.keras.layers.Dense(1, name='return_1')(combined),\n        'return_5': tf.keras.layers.Dense(1, name='return_5')(combined),\n        'direction_1': tf.keras.layers.Dense(3, activation='softmax', name='direction_1')(combined)}\n    model = tf.keras.Model(inputs, outputs, name='stock_lstm_v4_one_session_priority')\n    weights = dict(config['loss_weights'])\n    if variant == 'regression_only':\n        weights['direction_1'] = 0.\n    model.compile(optimizer=tf.keras.optimizers.Adam(config['learning_rate'], clipnorm=1.),\n        loss={'return_1': tf.keras.losses.Huber(delta=config['huber_delta']),\n            'return_5': tf.keras.losses.Huber(delta=config['huber_delta']),\n            'direction_1': tf.keras.losses.SparseCategoricalCrossentropy()}, loss_weights=weights,\n        metrics={'return_1': ['mae'], 'return_5': ['mae'], 'direction_1': ['accuracy']})\n    return model\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--data', type=Path, required=True)\n    parser.add_argument('--protocol', type=Path, required=True)\n    parser.add_argument('--output', type=Path, required=True)\n    parser.add_argument('--pilot', action='store_true')\n    parser.add_argument('--resume', action='store_true')\n    parser.add_argument('--variant', choices=['priority_multitask', 'regression_only'], default='priority_multitask')\n    args = parser.parse_args(argv)\n    import numpy as np\n    import pandas as pd\n    import tensorflow as tf\n    from stock_multi_agent.services.ml_dataset_v3 import sha256_file\n    from stock_multi_agent.services.ml_features_v3 import window_endpoints, window_batch, FEATURE_VERSION as V3_FEATURE_VERSION, FEATURES as V3_FEATURES\n    from stock_multi_agent.services.ml_validation import TrainOnlyScaler, regression_metrics, paired_date_block_bootstrap\n    from stock_multi_agent.services.ml_model_registry import GATE_IMPLEMENTATION_VERSION, evaluate_model_gate, research_gate_diagnostics\n    from stock_multi_agent.services.ml_v4 import (FEATURES, FEATURE_VERSION, feature_frame, eligible_session_folds,\n        origin_scales, direction_labels, direction_class_weights, split_calibration, fit_calibration,\n        apply_calibration, classification_metrics, baseline_predictions, scaler_specification)\n\n    protocol = json.loads(args.protocol.read_text(encoding='utf-8'))\n    if protocol.get('feature_version') != FEATURE_VERSION or protocol.get('horizons') != [1, 5] or protocol.get('window') != 30:\n        raise ValueError('V4 protocol/feature mismatch')\n    configuration = protocol['training']\n    manifest_path = args.data.parent / 'dataset.manifest.json'\n    metadata = json.loads(manifest_path.read_text(encoding='utf-8'))\n    dataset_hash = sha256_file(args.data)\n    if metadata.get('schema_version') != 'lstm-dataset-v3' or metadata.get('feature_version') != V3_FEATURE_VERSION or metadata.get('features') != V3_FEATURES or metadata['files']['dataset']['filename'] != args.data.name or dataset_hash != metadata['files']['dataset']['sha256']:\n        raise ValueError('Attach the unchanged V3 CSV with its matching original manifest')\n    if args.output.exists() and not args.resume:\n        raise FileExistsError('Use a new output folder, or resume the original config explicitly')\n    if args.resume and not (args.output / 'run-config.json').is_file():\n        raise ValueError('Resume requires original run-config.json and checkpoints')\n    args.output.mkdir(parents=True, exist_ok=True)\n    lock = {'dataset_sha256': dataset_hash, 'protocol_sha256': sha256_file(args.protocol),\n        'trainer_sha256': sha256_file(Path(__file__)), 'feature_version': FEATURE_VERSION,\n        'gate_implementation_version': GATE_IMPLEMENTATION_VERSION,\n        'runtime_sha256': os.getenv('STOCK_AGENT_V4_RUNTIME_SHA256', 'LOCAL_SOURCE'),\n        'pilot': args.pilot, 'variant': args.variant, 'tensorflow': tf.__version__, 'numpy': np.__version__}\n    lock_path = args.output / 'run-config.json'\n    if lock_path.exists() and json.loads(lock_path.read_text(encoding='utf-8')) != lock:\n        raise ValueError('Dataset/code/runtime/protocol/config changed: start a new experiment')\n    lock_path.write_text(json.dumps(lock, indent=2), encoding='utf-8')\n    if (args.output / 'manifest.json').exists():\n        raise FileExistsError('Run already finished; do not retune/rewrite the completed result')\n    print('GPU:', tf.config.list_physical_devices('GPU'), '| V4 feature build; no training yet.', flush=True)\n    frame = feature_frame(pd.read_csv(args.data, parse_dates=['date', 'target_end_1', 'target_end_5']), reuse_verified_v3_features=True)\n    endpoints = window_endpoints(frame, protocol['window'])\n    folds = eligible_session_folds(frame, endpoints, folds=protocol['outer_folds'], gap_sessions=protocol['gap_sessions'], minimum_sessions=protocol['minimum_eligible_sessions'])\n    targets = frame[['target_return_1', 'target_return_5']].to_numpy(dtype=np.float64)\n    scales = origin_scales(frame, floor=configuration['volatility_floor'], ceiling=configuration['volatility_ceiling'])\n    normalized = np.clip(targets / scales, -configuration['training_target_clip'], configuration['training_target_clip']).astype(np.float32)\n    # Warm-up/unlabelled rows cannot be batched; only known finite endpoints are used.\n    classes = np.ones(len(frame), dtype=np.int32)\n    classes[endpoints] = direction_labels(targets[endpoints, 0])\n    report = {'schema_version': 'lstm-evaluation-v4', 'protocol_id': protocol['protocol_id'],\n        'dataset_sha256': dataset_hash, 'scope': 'DEVELOPMENT_AFTER_V3_REVIEW_NOT_INDEPENDENT_HOLDOUT',\n        'pilot': bool(args.pilot or metadata.get('pilot_row_limit')), 'variant': args.variant,\n        'metric_precision': 'float64', 'gate_implementation_version': GATE_IMPLEMENTATION_VERSION,\n        'price_adjustment': 'ADJUSTED' if frame.price_adjustment.eq('ADJUSTED').all() else 'UNKNOWN',\n        'engineering_checks': {}, 'forward': {}, 'folds': [], 'splits': [fold.manifest() for fold in folds],\n        'feature_count': len(FEATURES), 'endpoint_count': len(endpoints),\n        'limitations': protocol['limitations'] + ['No automatic V4 integration; software tests do not constitute reviewed model engineering evidence.', 'Classifier probabilities are uncalibrated, not trade confidence.']}\n    print('Features:', len(FEATURES), '| endpoints:', len(endpoints), flush=True)\n    for split in report['splits']:\n        print('SPLIT:', json.dumps(split), flush=True)\n\n    def predict(model, values, indices):\n        pieces = {name: [] for name in ('return_1', 'return_5', 'direction_1')}\n        for offset in range(0, len(indices), configuration['batch_size']):\n            batch = indices[offset:offset + configuration['batch_size']]\n            result = model.predict_on_batch(window_batch(values, batch, protocol['window']))\n            if not isinstance(result, dict):\n                raise ValueError('Expected named V4 outputs, not positional guesses')\n            for name in pieces:\n                pieces[name].append(np.asarray(result[name], dtype=np.float64))\n        result = {name: np.concatenate(value) for name, value in pieces.items()}\n        result['returns'] = np.column_stack([result['return_1'][:, 0], result['return_5'][:, 0]]) * scales[indices]\n        if not all(np.isfinite(value).all() for value in result.values()):\n            raise ValueError('Nonfinite model outputs')\n        return result\n\n    class Batches(tf.keras.utils.Sequence):\n        def __init__(self, values, indices, class_weights, *, shuffle=False, seed=42):\n            super().__init__()\n            self.values, self.indices, self.class_weights = values, np.array(indices), class_weights\n            self.shuffle, self.rng = shuffle, np.random.default_rng(seed)\n            self.on_epoch_end()\n        def __len__(self):\n            return int(np.ceil(len(self.indices) / configuration['batch_size']))\n        def __getitem__(self, number):\n            indices = self.indices[number * configuration['batch_size']:(number + 1) * configuration['batch_size']]\n            ones = np.ones(len(indices), dtype=np.float32)\n            labels = {'return_1': normalized[indices, [0]][:, None], 'return_5': normalized[indices, [1]][:, None], 'direction_1': classes[indices]}\n            weights = {'return_1': ones, 'return_5': ones, 'direction_1': self.class_weights[classes[indices]]}\n            return window_batch(self.values, indices, protocol['window']), labels, weights\n        def on_epoch_end(self):\n            if self.shuffle:\n                self.rng.shuffle(self.indices)\n\n    class RawValidationMAE(tf.keras.callbacks.Callback):\n        def __init__(self, values, indices):\n            super().__init__()\n            self.values, self.indices = values, indices\n        def on_epoch_end(self, epoch, logs=None):\n            result = predict(self.model, self.values, self.indices)\n            score = regression_metrics(targets[self.indices, 0], result['returns'][:, 0])['mae']\n            if logs is not None:\n                logs['val_raw_return_1_mae'] = score\n            print(f' | raw next-session validation MAE: {score:.8f}', flush=True)\n\n    started, selected, best_selection = time.perf_counter(), None, float('inf')\n    seeds = [protocol['seeds'][0]] if args.pilot else protocol['seeds']\n    for fold in folds:\n        scaler = TrainOnlyScaler().fit(frame.iloc[fold.train][FEATURES].to_numpy(float))\n        values = scaler.transform(frame[FEATURES].to_numpy(float))\n        fit_indices, interval_indices = split_calibration(frame, fold.calibration)\n        baselines = {h: baseline_predictions(frame, fold, h) for h in protocol['horizons']}\n        class_weights = direction_class_weights(classes[fold.train])\n        for seed in seeds:\n            tf.keras.backend.clear_session()\n            tf.keras.utils.set_random_seed(seed)\n            trial = args.output / f'fold-{fold.fold_id}-seed-{seed}'\n            trial.mkdir(exist_ok=True)\n            checkpoint, result_path = trial / 'best.keras', trial / 'results.json'\n            if args.resume and result_path.exists():\n                saved = json.loads(result_path.read_text(encoding='utf-8'))\n                if saved['checkpoint_sha256'] != sha256_file(checkpoint) or saved['predictions_sha256'] != sha256_file(trial / 'predictions.csv'):\n                    raise ValueError('Completed trial checkpoint/predictions changed')\n                report['folds'].extend(saved['results'])\n                if fold.fold_id == folds[-1].fold_id and saved['selection_mae'] < best_selection:\n                    best_selection, selected = saved['selection_mae'], saved['spec']\n                    import shutil\n                    shutil.copy2(checkpoint, args.output / 'model.keras')\n                continue\n            model = tf.keras.models.load_model(checkpoint, safe_mode=True) if args.resume and checkpoint.exists() else build_model(tf, protocol['window'], len(FEATURES), configuration, variant=args.variant)\n            resume_score = None\n            if args.resume and checkpoint.exists():\n                resume_score = regression_metrics(targets[fold.validation, 0], predict(model, values, fold.validation)['returns'][:, 0])['mae']\n            train_indices = fold.train\n            if args.pilot and len(train_indices) > 50000:\n                train_indices = np.random.default_rng(seed).choice(train_indices, 50000, replace=False)\n            monitor = 'val_raw_return_1_mae'\n            callbacks = [RawValidationMAE(values, fold.validation),\n                tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor=monitor, mode='min', save_best_only=True, initial_value_threshold=resume_score),\n                tf.keras.callbacks.EarlyStopping(monitor=monitor, mode='min', baseline=resume_score, patience=configuration['early_stopping_patience'], restore_best_weights=True),\n                tf.keras.callbacks.ReduceLROnPlateau(monitor=monitor, mode='min', patience=3, factor=.5, min_lr=1e-5)]\n            print('Training fold/seed:', fold.fold_id, seed, flush=True)\n            history = model.fit(Batches(values, train_indices, class_weights, shuffle=True, seed=seed),\n                validation_data=Batches(values, fold.validation, np.ones(3, dtype=np.float32)),\n                epochs=3 if args.pilot else configuration['epochs'], callbacks=callbacks, verbose=2)\n            pd.DataFrame(history.history).to_csv(trial / 'history.csv', index=False)\n            model = tf.keras.models.load_model(checkpoint, safe_mode=True)\n            calibration_raw = predict(model, values, fit_indices)\n            interval_raw = predict(model, values, interval_indices)\n            # All choices above were frozen before looking at this trial's outer test.\n            test_raw = predict(model, values, fold.test)\n            trial_rows, csv_rows, calibration = [], [], {}\n            for column, horizon in enumerate(protocol['horizons']):\n                fitted = fit_calibration(targets[fit_indices, column], calibration_raw['returns'][:, column],\n                    calibration_raw['direction_1'][:, 1], allow_flat_gate=horizon == 1 and args.variant != 'regression_only')\n                interval_prediction = apply_calibration(interval_raw['returns'][:, column], interval_raw['direction_1'][:, 1], fitted)\n                radius = float(np.quantile(np.abs(targets[interval_indices, column] - interval_prediction), .9, method='higher'))\n                calibration[str(horizon)] = {**fitted, 'absolute_residual_quantile_90': radius,\n                    'interval_fit_scope': 'PURGED_DISJOINT_CALIBRATION_INTERVAL_BLOCK',\n                    'calibration_fit_samples': len(fit_indices), 'interval_samples': len(interval_indices)}\n                prediction = apply_calibration(test_raw['returns'][:, column], test_raw['direction_1'][:, 1], fitted)\n                actual = targets[fold.test, column]\n                references, baseline_details = baselines[horizon]\n                train_signs = np.sign(targets[fold.train, column])\n                majority = 1 if np.count_nonzero(train_signs > 0) >= np.count_nonzero(train_signs < 0) else -1\n                metrics = regression_metrics(actual, prediction, train_direction=majority)\n                reference_metrics = {name: regression_metrics(actual, value, train_direction=majority) for name, value in references.items()}\n                row = {'fold_id': fold.fold_id, 'seed': seed, 'horizon': horizon, **metrics,\n                    'zero_mae': reference_metrics['zero']['mae'], 'ridge_mae': reference_metrics['ridge']['mae'],\n                    'tree_mae': reference_metrics.get('tree', {}).get('mae'), 'tree_status': baseline_details['tree_status'],\n                    'all_baselines': reference_metrics, 'calibration_alpha': fitted['alpha'],\n                    'flat_threshold': fitted['flat_threshold'], 'uses_lstm_signal': fitted['alpha'] > 0 and bool(np.ptp(test_raw['returns'][:, column]) > 1e-12),\n                    'mae_improvement_ci': paired_date_block_bootstrap(actual, prediction, references['zero'], frame.iloc[fold.test].date.to_numpy(),\n                        **{name: protocol['bootstrap'][name] for name in ('block_sessions', 'repetitions')}),\n                    'interval_coverage': float(np.mean(np.abs(actual - prediction) <= radius)), 'interval_average_width_return': 2 * radius,\n                    'interval_status': 'EMPIRICAL_DISJOINT_CALIBRATION_NO_FUTURE_COVERAGE_GUARANTEE',\n                    'split': fold.manifest(), 'subgroups': {}}\n                for label, mask in {'active100': frame.iloc[fold.test].active_ratio_20.eq(1).to_numpy(),\n                    'low_activity': frame.iloc[fold.test].active_ratio_20.lt(1).to_numpy(), 'flat_target': np.abs(actual) <= 1e-12,\n                    'nonflat_target': np.abs(actual) > 1e-12}.items():\n                    row['subgroups'][label] = regression_metrics(actual[mask], prediction[mask], train_direction=majority) if mask.any() else None\n                if horizon == 1:\n                    row['direction_head'] = classification_metrics(actual, test_raw['direction_1'])\n                trial_rows.append(row)\n                part = frame.iloc[fold.test][['ticker', 'date', 'target_end_1', 'target_end_5', 'active_ratio_20']].copy()\n                part['actual'], part['prediction'], part['raw_prediction'] = actual, prediction, test_raw['returns'][:, column]\n                part['horizon'], part['fold_id'], part['seed'] = horizon, fold.fold_id, seed\n                part['flat_probability'] = test_raw['direction_1'][:, 1]\n                part['majority_direction'] = majority\n                for name, value in references.items():\n                    part[name] = value\n                csv_rows.append(part)\n            pd.concat(csv_rows).to_csv(trial / 'predictions.csv', index=False)\n            selection_mae = float(min([*history.history[monitor], *([] if resume_score is None else [resume_score])]))\n            spec = {'schema_version': 'lstm-spec-v4', 'model_version': 'lstm-one-session-priority-v4',\n                'protocol_id': protocol['protocol_id'], 'feature_version': FEATURE_VERSION, 'features': FEATURES,\n                'window': protocol['window'], 'horizons': protocol['horizons'], 'scaler': scaler_specification(scaler),\n                'origin_scale': {'feature': 'volatility_20', 'floor': configuration['volatility_floor'], 'ceiling': configuration['volatility_ceiling'], 'horizon_multiplier': [1., float(np.sqrt(5.))]},\n                'output_heads': ['return_1', 'return_5', 'direction_1'], 'class_order': ['DOWN', 'FLAT', 'UP'],\n                'calibration': calibration, 'seed': seed, 'selected_fold': fold.fold_id,\n                'selected_using': 'LATEST_FOLD_RAW_ONE_SESSION_VALIDATION_MAE_ONLY', 'dataset_sha256': dataset_hash,\n                'max_gap_business_days': metadata['max_gap_business_days'],\n                'price_adjustment': report['price_adjustment'], 'price_unit': str(frame.price_unit.iloc[0]) if frame.price_unit.nunique() == 1 else 'MIXED_UNKNOWN'}\n            result_path.write_text(json.dumps({'results': trial_rows, 'selection_mae': selection_mae, 'spec': spec,\n                'checkpoint_sha256': sha256_file(checkpoint), 'predictions_sha256': sha256_file(trial / 'predictions.csv')}, indent=2, allow_nan=False), encoding='utf-8')\n            report['folds'].extend(trial_rows)\n            if fold.fold_id == folds[-1].fold_id and selection_mae < best_selection:\n                best_selection, selected = selection_mae, spec\n                import shutil\n                shutil.copy2(checkpoint, args.output / 'model.keras')\n    report['elapsed_seconds'] = time.perf_counter() - started\n    report['gates_by_horizon'] = {str(h): evaluate_model_gate(report, protocol, horizon=h).as_dict() for h in protocol['horizons']}\n    report['research_diagnostics'] = {str(h): research_gate_diagnostics(report, protocol, horizon=h) for h in protocol['horizons']}\n    # V4 is deliberately NOT an app-serving artifact. Preserve research gate\n    # results but do not equate successful serialization with integration.\n    for gate in report['gates_by_horizon'].values():\n        gate.update(policy_eligibility=False, technical_blend_weight=0.)\n    for name, value in [('spec.json', selected), ('evaluation.json', report), ('protocol.json', protocol)]:\n        (args.output / name).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')\n    import shutil\n    shutil.copy2(Path(__file__), args.output / 'training-source.py')\n    runtime_path = os.getenv('STOCK_AGENT_V4_RUNTIME_PATH', '')\n    if runtime_path:\n        if sha256_file(Path(runtime_path)) != lock['runtime_sha256']:\n            raise ValueError('Runtime bundle changed during training')\n        shutil.copy2(runtime_path, args.output / 'runtime-source.zip')\n    files = {str(path.relative_to(args.output)).replace('\\\\', '/'): sha256_file(path) for path in args.output.rglob('*') if path.is_file()}\n    artifact = {'schema_version': 'lstm-artifact-v4', 'model_version': 'lstm-one-session-priority-v4',\n        'files': files, 'artifact_identity': hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest(),\n        'predictor_identity': hashlib.sha256(json.dumps({name: files[name] for name in ('model.keras', 'spec.json', 'protocol.json')}, sort_keys=True).encode()).hexdigest(),\n        'default_policy': 'EXPERIMENTAL_SHADOW_ZERO_WEIGHT_NO_AUTO_INTEGRATION',\n        'prediction_csvs_pinned': True, 'code_config': lock}\n    (args.output / 'manifest.json').write_text(json.dumps(artifact, indent=2), encoding='utf-8')\n    print(json.dumps({'gates_by_horizon': report['gates_by_horizon'], 'output': str(args.output), 'elapsed_seconds': report['elapsed_seconds']}, indent=2), flush=True)\n\n\nif __name__ == '__main__':\n    main()\n"

PROTOCOL_TEXT = "{\n  \"schema_version\": \"lstm-v4-protocol-1\",\n  \"protocol_id\": \"one-session-priority-v4-2026-10-18\",\n  \"frozen_at\": \"2026-10-18T03:39:33+00:00\",\n  \"registration_note\": \"Designed after reviewing V3 historical results. All historical folds are development-only; no claim of a new independent holdout.\",\n  \"feature_version\": \"single-ticker-v4-1\",\n  \"window\": 30,\n  \"horizons\": [1, 5],\n  \"outer_folds\": 3,\n  \"seeds\": [42, 123],\n  \"gap_sessions\": 5,\n  \"minimum_eligible_sessions\": {\"train\": 120, \"validation\": 30, \"calibration\": 30, \"test\": 40},\n  \"primary_metric\": \"raw_return_mae_horizon_1\",\n  \"selection\": \"raw one-session validation MAE only; no test-based seed or checkpoint selection\",\n  \"architecture\": \"LSTM32 plus last-observation skip, Dense32, separate 1/5 return heads and 3-class direction head\",\n  \"training\": {\n    \"epochs\": 35, \"learning_rate\": 0.0003, \"batch_size\": 256,\n    \"huber_delta\": 0.5, \"loss_weights\": {\"return_1\": 1.0, \"return_5\": 0.35, \"direction_1\": 0.15},\n    \"volatility_floor\": 0.005, \"volatility_ceiling\": 0.1,\n    \"training_target_clip\": 8.0, \"dropout\": 0.1, \"l2\": 0.00001,\n    \"early_stopping_patience\": 7\n  },\n  \"calibration\": {\"alpha_grid\": [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0], \"flat_threshold_grid\": [null,0.5,0.65,0.8], \"fit_fraction\": 0.65, \"interval_quantile\": 0.9, \"purge_fit_labels_before_interval\": true},\n  \"bootstrap\": {\"block_sessions\": 5, \"repetitions\": 500, \"confidence\": 0.95},\n  \"gate\": {\n    \"minimum_outer_folds\": 3, \"minimum_winning_folds\": 2,\n    \"max_relative_loss_vs_ridge_tree\": 0.01, \"minimum_forward_sessions\": 40,\n    \"require_adjusted_prices_for_blend\": true, \"maximum_technical_weight\": 0.2\n  },\n  \"default_policy\": \"EXPERIMENTAL_SHADOW_ZERO_WEIGHT_NO_AUTO_INTEGRATION\",\n  \"ablation\": \"Optional regression_only variant has direction loss weight 0; register output separately and do not choose a winner using outer test.\",\n  \"limitations\": [\"Current source universe is not point-in-time\", \"Adjusted-price and price-unit status remain UNKNOWN unless source evidence verifies them\", \"No headline accuracy threshold or baseline gate was relaxed to force a pass\"]\n}\n"


In [ ]:
# Runtime từ notebook, không phụ thuộc ZIP bị Kaggle tự giải nén.
source_hash = hashlib.sha256(json.dumps(PORTABLE_SOURCES, sort_keys=True).encode()).hexdigest()
if 'stock_multi_agent' in sys.modules and os.environ.get('STOCK_AGENT_V4_SOURCE_HASH') != source_hash:
    raise RuntimeError('Session giữ code V3/phiên bản khác. Restart session rồi Run All V4.')
staging = OUTPUT_ROOT / ('v4-notebook-runtime-' + RUN_ID)
staging.mkdir(parents=True, exist_ok=False)
runtime = staging / 'v4-runtime.zip'
with zipfile.ZipFile(runtime, 'x') as archive:
    for name in sorted(PORTABLE_SOURCES):
        entry = zipfile.ZipInfo(name, date_time=(2026, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(entry, PORTABLE_SOURCES[name])
with zipfile.ZipFile(runtime) as archive:
    assert all(archive.read(name).decode('utf-8') == value for name, value in PORTABLE_SOURCES.items())
os.environ['STOCK_AGENT_V4_SOURCE_HASH'] = source_hash
os.environ['STOCK_AGENT_V4_RUNTIME_SHA256'] = hashlib.sha256(runtime.read_bytes()).hexdigest()
os.environ['STOCK_AGENT_V4_RUNTIME_PATH'] = str(runtime)
sys.path.insert(0, str(runtime))
trainer_path = staging / 'train_lstm_kaggle_v4.py'
protocol_path = staging / 'lstm-v4-protocol.json'
trainer_path.write_text(TRAINING_SOURCE, encoding='utf-8')
protocol_path.write_text(PROTOCOL_TEXT, encoding='utf-8')
training = runpy.run_path(str(trainer_path), run_name='v4_training_import')
if EXPLICIT_CSV:
    data_path = Path(EXPLICIT_CSV).resolve()
    if not data_path.is_relative_to(INPUT_ROOT.resolve()) or not data_path.is_file():
        raise ValueError('EXPLICIT_CSV phải thuộc input đã attach.')
else:
    matches = sorted(INPUT_ROOT.rglob('vci_ohlcv_lstm_v3.csv'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Tìm thấy {len(matches)} CSV V3. Attach đúng dataset hoặc điền EXPLICIT_CSV.')
    data_path = matches[0]
if not (data_path.parent / 'dataset.manifest.json').is_file():
    raise FileNotFoundError('CSV phải đi cùng dataset.manifest.json gốc.')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('Data:', data_path, '| TensorFlow:', tf.__version__, '| GPUs:', gpus)
if not gpus:
    raise RuntimeError('Bật GPU trong notebook Settings rồi restart session.')
print('Sẵn sàng: cell tiếp theo kiểm checksum CSV và in split counts trước khi train.')


In [ ]:
if RESUME_DIR:
    RUN_DIR = Path(RESUME_DIR).resolve()
    if not RUN_DIR.is_relative_to(OUTPUT_ROOT.resolve()) or not (RUN_DIR / 'run-config.json').is_file():
        raise ValueError('Resume cần đúng run cũ dưới output với config/checkpoints.')
else:
    RUN_DIR = OUTPUT_ROOT / ('lstm-v4-' + RUN_MODE + '-' + RUN_ID)
arguments = ['--data', str(data_path), '--protocol', str(protocol_path), '--output', str(RUN_DIR), '--variant', VARIANT]
if RUN_MODE == 'pilot':
    arguments += ['--pilot']
if RESUME_DIR:
    arguments += ['--resume']
print('Output:', RUN_DIR)
training['main'](arguments)


In [ ]:
from IPython.display import FileLink, display
report_path = RUN_DIR / 'evaluation.json'
if not report_path.is_file() or not (RUN_DIR / 'manifest.json').is_file():
    raise FileNotFoundError('Run chưa hoàn tất. Xem lỗi cell training; không giả kết quả.')
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report['gates_by_horizon'], ensure_ascii=False, indent=2))
for horizon, result in report['research_diagnostics'].items():
    print('Horizon', horizon, '| winning folds:', result['winning_folds'], '| development passed:', result['passed'])
print('Pilot:', report['pilot'], '| scope:', report['scope'])
result_zip = OUTPUT_ROOT / (RUN_DIR.name + '-results.zip')
if result_zip.exists():
    raise FileExistsError('Archive đã tồn tại: tải bản cũ hoặc chọn run mới.')
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Tải ở Output/link bên dưới; lưu về D, không ghi đè trainResult/V2.')
display(FileLink(str(result_zip.relative_to(OUTPUT_ROOT))))
print('Archive:', result_zip)
print('V4 vẫn experimental/shadow; không tự đổi Technical score hoặc kết luận app.')


## Đọc kết quả

V4 thêm8feature causal; LSTM32 + nhánh quan sát cuối; riêng hai head return và head tăng/đứng/giảm; checkpoint theo raw next-session validation MAE. Calibration và interval-fit có purge/tách riêng. `evaluation.json` có gate1/5phiên, baseline/CI/subgroup và metric phân loại; `spec.json` pin feature/scaler/origin-volatility/calibration. Manifest pin cả predictions/checkpoints/code.

Giữ alpha0 khi calibration không tìm được tín hiệu, không ép model thắng. Classifier probabilities chưa được calibration, không phải confidence đầu tư. Range residual không bảo đảm coverage tương lai. V3/V4 đổi splits nên không so headline MAE trực tiếp. Giá điều chỉnh/unit UNKNOWN không tự được sửa. Chưa có bằng chứng V4 tốt hơn trước GPU training và forward evaluation.

Nguồn: [MAE và median](https://scikit-learn.org/stable/auto_examples/linear_model/plot_quantile_regression.html), [Keras multi-output](https://keras.io/guides/training_with_built_in_methods/), [LSTM GPU](https://keras.io/api/layers/recurrent_layers/lstm/).
